In [8]:
file = open("Billions Words/train_v2.txt")
number_of_lines = 100
for i in range(number_of_lines):
    line = file.readline()
    print(line)

The U.S. Centers for Disease Control and Prevention initially advised school systems to close if outbreaks occurred , then reversed itself , saying the apparent mildness of the virus meant most schools and day care centers should stay open , even if they had confirmed cases of swine flu .

When Ms. Winfrey invited Suzanne Somers to share her controversial views about bio-identical hormone treatment on her syndicated show in 2009 , it won Ms. Winfrey a rare dollop of unflattering press , including a Newsweek cover story titled " Crazy Talk : Oprah , Wacky Cures & You . "

Elk calling -- a skill that hunters perfected long ago to lure game with the promise of a little romance -- is now its own sport .

Don 't !

Fish , ranked 98th in the world , fired 22 aces en route to a 6-3 , 6-7 ( 5 / 7 ) , 7-6 ( 7 / 4 ) win over seventh-seeded Argentinian David Nalbandian .

Why does everything have to become such a big issue ?

AMMAN ( Reuters ) - King Abdullah of Jordan will meet U.S. President Ba

In [2]:
import numpy as np
import keras
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from keras.datasets import imdb
from time import time
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import csc_matrix, csr_matrix, dok_array
import sys
import pickle

print(sys.version)
min_frequency = 1
NUM_WORDS=20000

# sentences = []
# with open("Billions Words/train_v2.txt", encoding='utf-8') as f:
#     for line in f:
#         sentences.append(line.strip())

f = open("train_v2.txt", encoding='utf-8')
sentences = f.read().split("\n")
f.close()

vectorizer_X = CountVectorizer(min_df=min_frequency, max_features=NUM_WORDS, binary=True)
X = vectorizer_X.fit_transform(sentences)

f_vectorizer_X = open("vectorizer_X.pickle", "wb")
pickle.dump(vectorizer_X, f_vectorizer_X, protocol=4)
f_vectorizer_X.close()

f_X = open("X.pickle", "wb")
pickle.dump(X, f_X, protocol=4)
f_X.close()


3.11.7 (tags/v3.11.7:fa7a6f2, Dec  4 2023, 19:24:49) [MSC v.1937 64 bit (AMD64)]


In [10]:
import pickle
import codecs
from collections import defaultdict
import os

pair_list = []

folder_path = 'datasets/men-short'

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):
        file_name_without_extension = os.path.splitext(filename)[0]
        folder_name = file_name_without_extension
        new_folder_path = os.path.join(folder_path, folder_name)
        os.makedirs(new_folder_path, exist_ok=True)

        fread_simlex=codecs.open(file_path, 'r', 'utf-8')
        pair_list = []

        line_number = 0
        for line in fread_simlex:
            if line_number > 0:
                tokens = line.split(',')
                word_i = tokens[1].lower()
                word_j = tokens[2].lower()
                score = float(tokens[3].replace('\n', ''))
                pair_list.append( ((word_i, word_j), score) )
            line_number += 1

        word1_list=[]
        word2_list=[]

        for (x,y) in pair_list:
            (word1, word2) = x
            word1_list.append(word1)
            word2_list.append(word2)

        new_file_path = os.path.join(new_folder_path, file_name_without_extension)

        output= open(new_file_path + '_word1.pkl', "wb")
        pickle.dump(word1_list, output)
        output.close()

        output= open(new_file_path + '_word2.pkl', "wb")
        pickle.dump(word2_list, output)
        output.close()


In [ ]:
import os
from time import time
import pickle
import numpy as np
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
target_similarity=defaultdict(list)
from contextlib import redirect_stdout
from tqdm import tqdm

folder_path = 'datasets'
dir_count = 0
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        dir_count += 1
progress_bar = tqdm(total=dir_count, desc="Running Datasets")

for folder_name in os.listdir(folder_path):
    if folder_name == 'rg-65':
        current_folder_path = os.path.join(folder_path, folder_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, folder_name)
            with open(files_start_name + '_word1.pkl', 'rb') as f:
                word1 = pickle.load(f)
            with open(files_start_name + '_word2.pkl', 'rb') as f:
                word2 = pickle.load(f)
            word_total= list(set(word1 + word2))

            f_vectorizer_X = open("vectorizer_X.pickle", "rb")
            vectorizer_X = pickle.load(f_vectorizer_X)
            f_vectorizer_X.close()

            f_X = open("X.pickle", "rb")
            X_csr = pickle.load(f_X)
            f_X.close()
            X_train = X_csr

            feature_names = vectorizer_X.get_feature_names_out()
            number_of_features = vectorizer_X.get_feature_names_out().shape[0]
            target_words=[]
            for i in word_total:
                if i in vectorizer_X.vocabulary_:
                    target_words.append(i)
            output= open(files_start_name + '_target.pkl', "wb")
            pickle.dump(target_words, output)
            output.close()

            output_active = np.empty(len(target_words), dtype=np.uint32)
            for i in range(len(target_words)):
                target_word = target_words[i]
                target_id = vectorizer_X.vocabulary_[target_word]
                output_active[i] = target_id

            clause_weight_threshold = 0
            number_of_examples = 2000
            accumulation = 25
            factor = 4
            clauses = 80
            T = factor*40
            s = 5.0
            epochs = 25
            involved_datasets = []
            involved_datasets.append(["Combined",0,1])

            with open(files_start_name + '_results.txt', 'w') as file, redirect_stdout(file):
                print("Loading dataset: " + folder_name)
                tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=True)
                    
                print("Epochs: %d" % epochs)
                print("Example: %d" % number_of_examples)
                print("Target words: %d" % len(target_words))
                print("Accumulation: %d" % accumulation)
                print("No of features: %d" % number_of_features)

                total_training = 0
                epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                for e in range(epochs):
                    start_training = time()
                    tm.documents_fit(X_train, number_of_examples=number_of_examples, involved_datasets=involved_datasets)
                    stop_training = time()
                    total_training = total_training + (stop_training-start_training)
                    epochs_progress_bar.update(1)
                epochs_progress_bar.close()

                seconds = total_training
                hours = seconds // 3600
                minutes = (seconds % 3600) // 60
                seconds = seconds % 60
                print(f"Training duration: {hours} hours, {minutes} minutes, {seconds} seconds")

                print("\n=====================================\nClauses\n=====================================")
                for j in range(clauses):
                    print("Clause #%-2d " % (j), end=' ')
                    weight = 0
                    weight = tm.get_weight(0, j)
                    print("W:%-5d " % (weight), end=' ')
                    l = [] 
                    related_literals = []
                    number_of_literals = 0 
                    for k in range(tm.clause_bank.number_of_literals):
                        if tm.get_ta_action(j, k) == 1:
                            number_of_literals = number_of_literals + 1
                            related_literals.append(k)
                            if k < tm.clause_bank.number_of_features:
                                l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                            else:
                                l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                    print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                    try:
                        print(" ^ ".join(l))
                    except UnicodeEncodeError:
                        print(" exception ")

                profile = np.empty((len(target_words), clauses))
                for i in range(len(target_words)):
                    weights = tm.get_weights(i)
                    profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)

                output= open(files_start_name + '_tm_weights.pkl', "wb")
                pickle.dump(profile, output)
                output.close()

                similarity = cosine_similarity(profile)
                print("\n=====================================\nWord Similarity\n=====================================")
                max_word_length = len(max(target_words, key=len))
                for i in range(len(target_words)):
                    sorted_index = np.argsort(-1*similarity[i,:])
                    min_similarity = 1.0
                    max_similarity = 0.0
                    word_similarity = []
                    for j in range(1, len(target_words)):
                        target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                        word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
                        if(min_similarity > similarity[i,sorted_index[j]]):
                            min_similarity = similarity[i,sorted_index[j]]
                        if(max_similarity < similarity[i,sorted_index[j]]):
                            max_similarity = similarity[i,sorted_index[j]]
                    output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
                    print(output_line, end='     ==> ')
                    print(word_similarity)

                output= open(files_start_name + '_profile_dict.pkl', "wb")
                pickle.dump(target_similarity, output)
                output.close()
                progress_bar.update(1)
progress_bar.close()